In [5]:
# 依存関係はノートブック内で変更せず、起動前にプロジェクトルートで同期する。
# uv sync --group archive
import sys

assert sys.version_info[:2] == (3, 12), sys.version
print(sys.version)


Found existing installation: torch 2.9.0
Uninstalling torch-2.9.0:
  Successfully uninstalled torch-2.9.0
Found existing installation: torchvision 0.15.2
Uninstalling torchvision-0.15.2:
  Successfully uninstalled torchvision-0.15.2
Found existing installation: torchaudio 2.9.0
Uninstalling torchaudio-2.9.0:
  Successfully uninstalled torchaudio-2.9.0
Looking in indexes: https://download.pytorch.org/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.4/190.4 MB 11.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 3.3 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyannote-audio 4.0.1 requires torch>=2.8.0, but you have torch 2.3.1+cpu which is incompatible.
pyannote-audio 4.0.1 requires torchaudio>=2.8.0, but you have torchaudio 2.3.1+cpu which is incompatible.


### torchの設定

In [6]:
import sys, types, torchaudio

def _noop(*a, **k): pass
if not hasattr(torchaudio, "set_audio_backend"):
    torchaudio.set_audio_backend = _noop
if not hasattr(torchaudio, "get_audio_backend"):
    torchaudio.get_audio_backend = lambda: "soundfile"
if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]
torchaudio.set_audio_backend("soundfile")

mod_backend = types.ModuleType("torchaudio.backend")
mod_common  = types.ModuleType("torchaudio.backend.common")
try:
    from torchaudio import AudioMetaData as _AMD   # 2.x ではここにある
except Exception:
    class _AMD:                                    # 念のための空クラス
        pass
mod_common.AudioMetaData = _AMD
sys.modules["torchaudio.backend"] = mod_backend
sys.modules["torchaudio.backend.common"] = mod_common


import soundfile as sf
if not hasattr(torchaudio, "info"):
    def _info(path):
        try:
            f = sf.SoundFile(path)
            sr = f.samplerate
            frames = len(f)
            channels = f.channels
            f.close()
            class Info:
                sample_rate = sr
                num_channels = channels
                num_frames = frames
            return Info()
        except Exception as e:
            raise RuntimeError(f"torchaudio.info fallback failed for {path}: {e}")
    torchaudio.info = _info

ImportError: cannot import name '_cuda' from 'torch._utils' (/root/MedWhisper/venv/lib/python3.11/site-packages/torch/_utils.py)

### pyannoteの設定

In [ ]:

from pyannote.audio import Pipeline

# ※ トークンは環境変数化推奨（HF_TOKEN など）
pipe = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    use_auth_token=__import__("os").environ["HF_TOKEN"]
)
# GPU使えるなら
# pipe = pipe.to("cuda")

print("pipeline ready ✓")


#### wavと各関数の設定


In [ ]:
# === Block 0: 共通ユーティリティ ===
import numpy as np, librosa, soundfile as sf, pyloudnorm as pyln, scipy.signal as sig, matplotlib.pyplot as plt
from pathlib import Path

# file path 設定
FILE_NAME = "20240827/右後_4回目_大野"
# 基本のパスを組み立てる
BASE_DIR = Path("/root/MedWhisper")
WAV      = BASE_DIR / f"{FILE_NAME}.wav"
OUTDIR   = Path(f"./work/{FILE_NAME}"); OUTDIR.mkdir(exist_ok=True, parents=True)


# メトリクス
def rms_db(x): return 20*np.log10(np.sqrt(np.mean(x**2))+1e-12)
def lufs(x, sr): return pyln.Meter(sr).integrated_loudness(x)
def lufs_norm(x, sr, target=-24.0):
    m = pyln.Meter(sr)
    return np.clip(pyln.normalize.loudness(x, m.integrated_loudness(x), target), -1.0, 1.0)

# 粗VAD（RMSベース）
def rough_vad_mask(x, sr, frame=0.025, hop=0.010):
    frm = int(sr*frame); hopN = int(sr*hop)
    E = librosa.feature.rms(y=x, frame_length=frm, hop_length=hopN).squeeze()
    thr = np.percentile(E, 70) * 0.6
    t = np.arange(len(E))*hop + frame/2
    return t, (E>thr).astype(float), E, float(thr)

# 明瞭帯ピーキングEQ
def peaking_eq(x, fs, f0=1500.0, Q=1.0, gain_db=3.0):
    A = 10**(gain_db/40.0)
    w0 = 2*np.pi*f0/fs
    alpha = np.sin(w0)/(2.0*Q)
    cw = np.cos(w0)
    b0 = 1 + alpha*A
    b1 = -2*cw
    b2 = 1 - alpha*A
    a0 = 1 + alpha/A
    a1 = -2*cw
    a2 = 1 - alpha/A
    b = np.array([b0, b1, b2]) / a0
    a = np.array([1.0, a1/a0, a2/a0])
    return sig.filtfilt(b, a, x)

print("Block 0 ready ✓")

In [ ]:
# === 置き換え：読み込み〜可視化（上段は非正規化・下段は従来通りRMS） ===
import soundfile as sf
# === パス設定（シンプル構成） ===
#WAV = "/root/MedWhisper/20240827/右後_4回目_大野.wav"
  # ★ WAVファイルの絶対パス
# 1) 非正規化で読み込み（int16のまま）
y_raw, sr = sf.read(WAV, dtype='int16', always_2d=False)  # y_raw: int16, -32768..32767
if y_raw.ndim == 2:  # ステレオならモノラル化（平均）
    y_raw = y_raw.mean(axis=1).astype('int16')

# 2) 解析用に float へ変換（-1..1 にスケール）
y = y_raw.astype('float32') / 32768.0

# 3) 以降は従来処理（RMSなど）
dur = len(y) / sr
print(f"sr={sr}, dur={dur:.2f}s, peak(norm)={float(np.max(np.abs(y))):.3f}, RMS(dBFS)={rms_db(y):.1f}, LUFS={lufs(y,sr):.1f}")

t_mask, mask, E, thr = rough_vad_mask(y, sr)
max_pts = 200_000
step = max(1, len(y_raw) // max_pts)  # 上段は y_raw ベースで間引き
y_plot = y_raw[::step]
t_plot = np.arange(len(y_plot)) * step / sr

# 4) 可視化
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

# 上段：非正規化の生PCM波形（ylim設定なし）
ax[0].plot(t_plot, y_plot, lw=0.5, color="black")
ax[0].set_ylabel("Amplitude [PCM int16]", fontsize=14)  # -32768..32767 相当
ax[0].grid(True, alpha=0.3)

# 下段：RMSのみ（正規化表示）
E_scaled = E / (E.max() + 1e-12)
ax[1].plot(t_mask, E_scaled, lw=1.0, color="blue", label="RMS (scaled)")
ax[1].set_xlabel("Time [s]", fontsize=14)
ax[1].set_ylabel("Normalized RMS", fontsize=14)
ax[1].set_ylim(0, 1)
ax[1].grid(True, alpha=0.3)
# ax[1].legend(loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
# === Block 1: 入力診断（全区間可視化） ===
y, sr = librosa.load(WAV, sr=16000, mono=True)
dur = len(y)/sr
print(f"sr={sr}, dur={dur:.2f}s, peak={np.max(np.abs(y)):.3f}, RMS(dBFS)={rms_db(y):.1f}, LUFS={lufs(y,sr):.1f}")

# 粗VAD（全体）
t_mask, mask, E, thr = rough_vad_mask(y, sr)
speech_like = float(mask.mean())
print(f"rough-VAD speech-like ratio={speech_like:.2%} (thr={thr:.5f})")

# 可視化（全区間）
max_pts = 200_000
step = max(1, len(y) // max_pts)
y_plot = y[::step]
t_plot = np.arange(len(y_plot)) * step / sr

fig, ax = plt.subplots(2,1, figsize=(12,5), sharex=True)
ax[0].plot(t_plot, y_plot, lw=0.5)
# ax[0].set_title("A) Original waveform (full duration, decimated for display)")
ax[0].set_ylabel("Amplitude [-]") #基準電圧を±1として正規化した値　基準電圧に対する日ということ
ax[0].grid(True, alpha=0.3)

# …前略（E, thr を計算済み）…

E_scaled = E / (E.max() + 1e-12)
thr_scaled = thr / (E.max() + 1e-12)  # ★ 閾値をE_scaledのスケールに合わせる

ax[1].plot(t_mask, E_scaled, lw=1.0, label="RMS (scaled)")
ax[1].step(t_mask, mask, where="post", label="rough-VAD mask", alpha=0.7)

# ★ 閾値ライン（赤の破線）
ax[1].axhline(y=thr_scaled, linestyle="--", linewidth=1.0, color="red", label=f"threshold ({thr_scaled:.2f})")

ax[1].set_xlabel("Time [s]")
ax[1].set_ylabel("Normalized RMS / mask")
ax[1].set_ylim(0,1)
ax[1].grid(True, alpha=0.3)
ax[1].legend(loc="upper right")

#### 学生データに対して

In [ ]:
# # === Block 0: 共通ユーティリティ ===
# import numpy as np, librosa, soundfile as sf, pyloudnorm as pyln, scipy.signal as sig, matplotlib.pyplot as plt
# from pathlib import Path

# # この1行を変更するだけで、すべてのファイルパスが切り替わる
# FILE_NAME = "0604data/7回目_右後ろ"

# # 基本のパスを組み立てる
# BASE_DIR = Path("/root/MedWhisper")
# WAV      = BASE_DIR / f"{FILE_NAME}.wav"
# OUTDIR   = Path(f"./_work/{FILE_NAME}"); OUTDIR.mkdir(exist_ok=True, parents=True)


# # メトリクス
# def rms_db(x): return 20*np.log10(np.sqrt(np.mean(x**2))+1e-12)
# def lufs(x, sr): return pyln.Meter(sr).integrated_loudness(x)
# def lufs_norm(x, sr, target=-24.0):
#     m = pyln.Meter(sr)
#     return np.clip(pyln.normalize.loudness(x, m.integrated_loudness(x), target), -1.0, 1.0)

# # 粗VAD（RMSベース）
# def rough_vad_mask(x, sr, frame=0.025, hop=0.010):
#     frm = int(sr*frame); hopN = int(sr*hop)
#     E = librosa.feature.rms(y=x, frame_length=frm, hop_length=hopN).squeeze()
#     thr = np.percentile(E, 70) * 0.6
#     t = np.arange(len(E))*hop + frame/2
#     return t, (E>thr).astype(float), E, float(thr)

# # 明瞭帯ピーキングEQ
# def peaking_eq(x, fs, f0=1500.0, Q=1.0, gain_db=3.0):
#     A = 10**(gain_db/40.0)
#     w0 = 2*np.pi*f0/fs
#     alpha = np.sin(w0)/(2.0*Q)
#     cw = np.cos(w0)
#     b0 = 1 + alpha*A
#     b1 = -2*cw
#     b2 = 1 - alpha*A
#     a0 = 1 + alpha/A
#     a1 = -2*cw
#     a2 = 1 - alpha/A
#     b = np.array([b0, b1, b2]) / a0
#     a = np.array([1.0, a1/a0, a2/a0])
#     return sig.filtfilt(b, a, x)

# print("Block 0 ready ✓")

In [ ]:
# # === Block 1: 入力診断（全区間可視化） ===
# y, sr = librosa.load(WAV, sr=16000, mono=True)
# dur = len(y)/sr
# print(f"sr={sr}, dur={dur:.2f}s, peak={np.max(np.abs(y)):.3f}, RMS(dBFS)={rms_db(y):.1f}, LUFS={lufs(y,sr):.1f}")

# # 粗VAD（全体）
# t_mask, mask, E, thr = rough_vad_mask(y, sr)
# speech_like = float(mask.mean())
# print(f"rough-VAD speech-like ratio={speech_like:.2%} (thr={thr:.5f})")

# # 可視化（全区間）
# max_pts = 200_000
# step = max(1, len(y) // max_pts)
# y_plot = y[::step]
# t_plot = np.arange(len(y_plot)) * step / sr

# fig, ax = plt.subplots(2,1, figsize=(12,5), sharex=True)
# ax[0].plot(t_plot, y_plot, lw=0.5)
# # ax[0].set_title("A) Original waveform (full duration, decimated for display)")
# ax[0].set_ylabel("Amplitude [-]") #基準電圧を±1として正規化した値　基準電圧に対する日ということ
# ax[0].grid(True, alpha=0.3)

# # …前略（E, thr を計算済み）…

# E_scaled = E / (E.max() + 1e-12)
# thr_scaled = thr / (E.max() + 1e-12)  # ★ 閾値をE_scaledのスケールに合わせる

# ax[1].plot(t_mask, E_scaled, lw=1.0, label="RMS (scaled)")
# ax[1].step(t_mask, mask, where="post", label="rough-VAD mask", alpha=0.7)

# # ★ 閾値ライン（赤の破線）
# ax[1].axhline(y=thr_scaled, linestyle="--", linewidth=1.0, color="red", label=f"threshold ({thr_scaled:.2f})")

# ax[1].set_xlabel("Time [s]")
# ax[1].set_ylabel("Normalized RMS / mask")
# ax[1].set_ylim(0,1)
# ax[1].grid(True, alpha=0.3)
# ax[1].legend(loc="upper right")

In [ ]:
# # === Pre-Whisper pipeline: denoise + VAD chunking ===
# import json, math
# from pathlib import Path
# import numpy as np, librosa, soundfile as sf, scipy.signal as sig
# import matplotlib.pyplot as plt  # プロット不要ならimport不要

# # -------------------- 設定 --------------------
# IN_WAV  = "/root/MedWhisper/0604data/7回目_右後ろ.wav"
# SR      = 16000
# FRAME   = 0.025
# HOP     = 0.010
# HPF_HZ  = 80
# LPF_HZ  = 8000
# PEAK_F0 = 1500.0
# PEAK_Q  = 1.0
# PEAK_DB = 3.0

# # VADパラメータ（ヒステリシス＋後処理）
# VAD_PCTL = 70         # RMS分布の上位%を基準
# VAD_GAIN = 0.6        # 基準×係数 => 低めに
# HYST_UP  = 1.2        # ONになる時は thr*HYST_UP
# HYST_DN  = 0.8        # OFFになる時は thr*HYST_DN
# MIN_SPEECH = 0.40     # 最小発話長[s]
# MIN_SIL    = 0.20     # 最小無音長[s]
# BRIDGE_GAP = 0.30     # この無音以下なら隣接発話を連結

# # 出力場所
# IN_P = Path(IN_WAV)
# OUTDIR = IN_P.parent / "_whisper_prep"
# SEGDIR = OUTDIR / "segments"
# OUTDIR.mkdir(parents=True, exist_ok=True)
# SEGDIR.mkdir(parents=True, exist_ok=True)

# # -------------------- ユーティリティ --------------------
# def lufs_norm(x, sr, target=-24.0):
#     # ライブラリ無しの簡易近似：RMSで揃える（LUFSがあれば置換可）
#     rms = np.sqrt(np.mean(np.square(x)) + 1e-12)
#     tgt = 10 ** (target/20.0)  # 目安（相対スケール）
#     if rms < 1e-6: return x
#     g = min(1.0, tgt / rms)
#     return np.clip(x * g, -1.0, 1.0)

# def peaking_eq(x, fs, f0=1500.0, Q=1.0, gain_db=3.0):
#     A = 10**(gain_db/40.0)
#     w0 = 2*np.pi*f0/fs
#     alpha = np.sin(w0)/(2.0*Q)
#     cw = np.cos(w0)
#     b0 = 1 + alpha*A
#     b1 = -2*cw
#     b2 = 1 - alpha*A
#     a0 = 1 + alpha/A
#     a1 = -2*cw
#     a2 = 1 - alpha/A
#     b = np.array([b0, b1, b2]) / a0
#     a = np.array([1.0, a1/a0, a2/a0])
#     return sig.filtfilt(b, a, x)

# def nz01(x):
#     m, M = np.nanpercentile(x, 5), np.nanpercentile(x, 95)
#     return np.clip((x-m)/(M-m+1e-12), 0, 1)

# # -------------------- 1) 読み込み＆基本整形 --------------------
# y, _ = librosa.load(IN_P.as_posix(), sr=SR, mono=True)
# frm, hopN = int(SR*FRAME), int(SR*HOP)

# # HPF/LPF（安定なSOS）→ 明瞭帯EQ → クリッピング回避
# nyq = SR/2
# LPF_SAFE = min(LPF_HZ, 0.95*nyq)
# sos = sig.butter(4, HPF_HZ, btype="highpass", fs=SR, output="sos")
# y = sig.sosfiltfilt(sos, y)
# sos = sig.butter(4, LPF_SAFE, btype="lowpass", fs=SR, output="sos")
# y = sig.sosfiltfilt(sos, y)
# y = peaking_eq(y, fs=SR, f0=PEAK_F0, Q=PEAK_Q, gain_db=PEAK_DB)
# pk = np.max(np.abs(y))+1e-12
# if pk > 0.99: y = y / (pk/0.99)

# # -------------------- 2) 局所品質で軽ノイズ抑制（悪所だけ） --------------------
# # 特徴（center=Falseで統一）
# RMS  = librosa.feature.rms(y=y, frame_length=frm, hop_length=hopN, center=False).squeeze()
# FLAT = librosa.feature.spectral_flatness(y=y, n_fft=1024, hop_length=hopN, center=False).squeeze()

# # HNRライク（自己相関、0-20ms ラグピーク）
# frames = librosa.util.frame(y, frame_length=frm, hop_length=hopN, axis=0)
# def hnr_like(frame):
#     f = frame - frame.mean()
#     ac = sig.correlate(f, f, mode="full")[len(f)-1:]
#     if ac[0] <= 1e-12: return np.nan
#     search = max(2, int(0.02*SR))
#     pk = np.max(ac[1:search]) if search < len(ac) else np.max(ac[1:])
#     return 10*np.log10((pk+1e-12)/(ac[0]+1e-12))
# HNR = np.array([hnr_like(fr) for fr in frames])

# noise_floor = np.percentile(RMS, 10) + 1e-12
# SNRdB = 20*np.log10((RMS+1e-12)/noise_floor)

# q_snr  = nz01(SNRdB)
# q_hnr  = nz01(np.nan_to_num(HNR, nan=np.nanmin(HNR)))
# q_flat = 1.0 - nz01(FLAT)
# L = min(len(q_snr), len(q_hnr), len(q_flat))
# Q = (0.5*q_snr[:L] + 0.3*q_hnr[:L] + 0.2*q_flat[:L])

# # STFT
# N_FFT = 1024
# S = librosa.stft(y, n_fft=N_FFT, hop_length=hopN, win_length=frm,
#                  window="hann", center=False)
# mag, ang = np.abs(S), np.angle(S)

# Q_GOOD, Q_BAD = 0.60, 0.50
# good = (Q >= Q_GOOD); bad = (Q <= Q_BAD)
# if np.any(bad):
#     Nspec = np.median(mag[:, bad], axis=1, keepdims=True)
# else:
#     Nspec = np.percentile(mag, 10, axis=1, keepdims=True)
# alpha = 1.0
# G = (mag**2)/(mag**2 + (alpha*Nspec)**2 + 1e-12)
# G = np.clip(G, 0.15, 1.0)
# G_local = G.copy()
# mid = (~good) & (~bad)
# if np.any(good): G_local[:, good] = np.maximum(G_local[:, good], 0.90)
# if np.any(mid):  G_local[:, mid]  = np.minimum(G_local[:, mid], 0.75)
# if np.any(bad):  G_local[:, bad]  = np.minimum(G_local[:, bad], 0.60)
# G_local = sig.medfilt(G_local, kernel_size=(1,5))

# S_hat = G_local*mag*np.exp(1j*ang)
# y_dnz = librosa.istft(S_hat, hop_length=hopN, win_length=frm,
#                       window="hann", center=False, length=len(y))
# # 軽く正規化
# pk = np.max(np.abs(y_dnz))+1e-12
# if pk > 0.99: y_dnz = y_dnz/(pk/0.99)

# # （任意）全体レベルを-24 LUFS目安に
# y_dnz = lufs_norm(y_dnz, SR, target=-24.0)

# # 保存（全体の前処理済み）
# FULL_OUT = OUTDIR / "denoised_full.wav"
# sf.write(FULL_OUT.as_posix(), y_dnz.astype(np.float32), SR, subtype="PCM_16")
# print("full saved:", FULL_OUT)

# # -------------------- 3) ヒステリシス付きRMS-VAD → チャンク化 --------------------
# # 再計算（denoised上で）
# RMS2 = librosa.feature.rms(y=y_dnz, frame_length=frm, hop_length=hopN, center=False).squeeze()
# thr_base = np.percentile(RMS2, VAD_PCTL) * VAD_GAIN
# thr_on, thr_off = thr_base*HYST_UP, thr_base*HYST_DN

# # ヒステリシス
# mask = np.zeros_like(RMS2, dtype=np.uint8)
# state = 0
# for i, e in enumerate(RMS2):
#     if state==0 and e>thr_on:  state=1
#     if state==1 and e<thr_off: state=0
#     mask[i]=state

# # 小島除去・短ギャップ連結
# def runs(bits):
#     idx = np.flatnonzero(np.diff(np.r_[0, bits, 0]) != 0)
#     lens = np.diff(idx)
#     vals = bits[idx[:-1]]
#     starts = idx[:-1][vals==1]
#     stops  = idx[1:][vals==1]
#     return starts, stops

# starts, stops = runs(mask)
# to_keep = []
# min_speech_f = int(MIN_SPEECH/HOP)
# min_sil_f    = int(MIN_SIL/HOP)
# bridge_f     = int(BRIDGE_GAP/HOP)

# # 連結処理
# i=0
# while i < len(starts):
#     s = starts[i]; e = stops[i]
#     # 以降の近接セグメントを連結
#     while i+1 < len(starts) and (starts[i+1]-e) <= bridge_f:
#         e = stops[i+1]; i += 1
#     # 短すぎる発話は捨てる
#     if (e - s) >= min_speech_f:
#         to_keep.append((s, e))
#     i += 1

# # 開始/終了の余白（安全マージン）
# PAD = 0.10  # s
# pad_f = int(PAD/HOP)

# segments = []
# for k, (s,e) in enumerate(to_keep, 1):
#     s0 = max(0, s - pad_f)
#     e0 = min(len(RMS2)-1, e + pad_f)
#     t0 = s0*HOP
#     t1 = e0*HOP + FRAME  # フレーム中心→末尾寄せ
#     i0 = int(t0*SR)
#     i1 = min(len(y_dnz), int(t1*SR))
#     wav_k = SEGDIR / f"seg_{k:04d}.wav"
#     sf.write(wav_k.as_posix(), y_dnz[i0:i1].astype(np.float32), SR, subtype="PCM_16")
#     segments.append({"id": k, "start": round(t0,3), "end": round(t1,3), "path": wav_k.as_posix()})

# MANIFEST = OUTDIR / "segments.jsonl"
# with open(MANIFEST, "w", encoding="utf-8") as f:
#     for s in segments:
#         f.write(json.dumps(s, ensure_ascii=False) + "\n")

# print(f"segments: {len(segments)}  -> {SEGDIR}")
# print("manifest:", MANIFEST)

# # -------------------- 4) （任意）可視化 --------------------
# # 閾値ラインとmaskを表示（品質確認）
# t_mask = np.arange(len(RMS2))*HOP + FRAME/2
# E_scaled = RMS2/(RMS2.max()+1e-12)
# thr_scaled = thr_base/(RMS2.max()+1e-12)

# plt.figure(figsize=(12,4))
# plt.plot(t_mask, E_scaled, lw=0.8, label="RMS (scaled)")
# plt.step(t_mask, mask.astype(float), where="post", label="VAD mask", alpha=0.6)
# plt.axhline(thr_scaled, color="r", ls="--", lw=1.0,
#             label=f"threshold ~ {thr_base:.4f}")
# plt.xlabel("Time [s]"); plt.ylabel("Normalized RMS / mask")
# plt.title("Energy & VAD (after denoise)")
# plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# === Quick Check: RMSヒストグラム＋可変閾値（W1:良い / W2:悪い） ===
import numpy as np, librosa, matplotlib.pyplot as plt

# ★ ここだけ指定
W1_PATH = "/root/MedWhisper/202502/右後ろ_1回目.wav"   # 良い
W2_PATH = "/root/MedWhisper/0604data/7回目_右後ろ.wav"    # 悪い
SR = 16000

def rms_series(y, sr, frame=0.025, hop=0.010):
    frm = int(sr*frame); hopN = int(sr*hop)
    E = librosa.feature.rms(y=y, frame_length=frm, hop_length=hopN).squeeze()
    thr = np.percentile(E, 70) * 0.6
    speech_like = float((E > thr).mean())
    return E, thr, speech_like

# load
y1, sr1 = librosa.load(W1_PATH, sr=SR, mono=True)
y2, sr2 = librosa.load(W2_PATH, sr=SR, mono=True)

# frame-RMS & threshold
E1, thr1, r1 = rms_series(y1, SR)
E2, thr2, r2 = rms_series(y2, SR)

# plot
plt.figure(figsize=(7,4))
bins = np.linspace(0, max(E1.max(), E2.max()), 100)
plt.hist(E1, bins=bins, alpha=0.55, label=f"W1 good  (duty={r1:.2f})")
plt.hist(E2, bins=bins, alpha=0.55, label=f"W2 bad   (duty={r2:.2f})")
plt.axvline(thr1, ls="--", c="C0", lw=1.5, label=f"W1 thr={thr1:.4f}")
plt.axvline(thr2, ls="--", c="C1", lw=1.5, label=f"W2 thr={thr2:.4f}")
plt.xlabel("Frame RMS"); plt.ylabel("Count")
plt.title("RMS histogram with adaptive thresholds")
plt.legend(); plt.tight_layout(); plt.show()

# 簡易SNR参考（下位10%をノイズ床とみなす）
nf1, nf2 = np.percentile(E1,10), np.percentile(E2,10)
snr1 = 20*np.log10((np.median(E1)+1e-12)/(nf1+1e-12))
snr2 = 20*np.log10((np.median(E2)+1e-12)/(nf2+1e-12))
print(f"[W1] thr={thr1:.5f} duty={r1:.2f} est.SNR≈{snr1:.1f} dB")
print(f"[W2] thr={thr2:.5f} duty={r2:.2f} est.SNR≈{snr2:.1f} dB")


#### 雑が気

In [ ]:
# === Block 2: 帯域整形＋明瞭帯EQ（全区間） ===
HPF_HZ   = 80       # 低域カット
LPF_HZ   = 8000     # 超高域カット
PEAK_F0  = 1500.0   # 明瞭帯
PEAK_Q   = 1.0
PEAK_DB  = 3.0

xB = y.copy()
nyq = sr / 2
LPF_SAFE = min(LPF_HZ, 0.95 * nyq)

# 安定な sos フィルタ
sos = sig.butter(4, HPF_HZ, btype="highpass", fs=sr, output="sos")
xB = sig.sosfiltfilt(sos, xB)
sos = sig.butter(4, LPF_SAFE, btype="lowpass",  fs=sr, output="sos")
xB = sig.sosfiltfilt(sos, xB)

# ピーキングEQ（+3dB）
xB = peaking_eq(xB, fs=sr, f0=PEAK_F0, Q=PEAK_Q, gain_db=PEAK_DB)
xB = np.clip(xB, -1, 1)

# 出力先のパスを先に変数として定義
output_path = OUTDIR / f"re2{FILE_NAME}.wav"

# ファイルを書き込む前に、親ディレクトリが存在するか確認し、なければ作成する
output_path.parent.mkdir(parents=True, exist_ok=True)

# 出力＆指標
sf.write(output_path.as_posix(), xB, sr) # 変数を使って書き込む
print(f"[B] HPF={HPF_HZ}Hz, LPF={int(LPF_SAFE)}Hz, Peak {PEAK_F0}Hz +{PEAK_DB}dB")
print(f"RMS(dBFS)={rms_db(xB):.1f}, LUFS={lufs(xB,sr):.1f}, peak={np.max(np.abs(xB)):.3f}")

# 可視化
max_pts = 200_000
stepA = max(1, len(y)  // max_pts)
stepB = max(1, len(xB) // max_pts)
tA = np.arange(0, len(y),  stepA) / sr
tB = np.arange(0, len(xB), stepB) / sr
y_d = y[ ::stepA]
b_d = xB[::stepB]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# 描画順を入れ替える: まずBを描画し、その上にAを描画する
axes[0].plot(tB, b_d, lw=0.5, label="B") # 先にBを描画
axes[0].plot(tA, y_d, lw=0.5, label="A") # その上にAを描画
axes[0].set_title("A vs B waveform (full duration, decimated)")
axes[0].set_xlabel("time [s]"); axes[0].set_ylabel("amplitude")
axes[0].grid(True, alpha=0.3); axes[0].legend()

def ltas_db(x, sr, n_fft=2048, hop=512):
    S = np.abs(librosa.stft(x, n_fft=n_fft, hop_length=hop)).mean(axis=1)
    f = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    return f, 20*np.log10(S + 1e-12)

fA, dA = ltas_db(y,  sr)
fB, dB = ltas_db(xB, sr)
dA_i = np.interp(fB, fA, dA)
axes[1].semilogx(fB, dB - dA_i, lw=1.0)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_xlim(40, 8000)
axes[1].set_ylabel("Δ dB (B - A)")
axes[1].set_xlabel("Hz")
axes[1].set_title("LTAS diff: B - A (full duration)")
axes[1].grid(True, which="both", alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
from collections import defaultdict

wav_for_diar = (OUTDIR / f"re2{FILE_NAME}.wav").as_posix()

# dia = pipe(wav_for_diar, min_speakers=1, max_speakers=3) # 人数指定
dia = pipe(wav_for_diar) # 人数自動推定

segments = []  # [(speaker, start, end)]
for turn, _, speaker in dia.itertracks(yield_label=True):
    segments.append((speaker, float(turn.start), float(turn.end)))

dur_by_spk = defaultdict(float)
for spk, s, e in segments:
    dur_by_spk[spk] += (e - s)

print("speakers:", list(dur_by_spk.keys()))
print("durations:", dict(dur_by_spk))
print("segments (head):", segments[:8])

In [ ]:
# === Block S3: 後処理（連結・除外・パディング）→ 話者別に書き出し ===
from itertools import groupby

def merge_close_segments(seg_list, max_gap=0.6, min_keep=0.8):
    """
    seg_list: [(spk, start, end)]
    - 同一spkで max_gap以下の隙間は連結
    - 連結後に min_keep 未満は除外
    """
    seg_list = sorted(seg_list, key=lambda x: (x[0], x[1], x[2]))
    merged = []
    for spk, group in groupby(seg_list, key=lambda x: x[0]):
        group = list(group)
        cur_s, cur_e = group[0][1], group[0][2]
        for _, s, e in group[1:]:
            if s - cur_e <= max_gap:
                cur_e = max(cur_e, e)
            else:
                if (cur_e - cur_s) >= min_keep:
                    merged.append((spk, cur_s, cur_e))
                cur_s, cur_e = s, e
        if (cur_e - cur_s) >= min_keep:
            merged.append((spk, cur_s, cur_e))
    merged.sort(key=lambda x: x[1])
    return merged

def pad_segments(seg_list, pad=0.25, total_dur=None):
    out = []
    for spk, s, e in seg_list:
        s2 = max(0.0, s - pad)
        e2 = e + pad if total_dur is None else min(total_dur, e + pad)
        out.append((spk, s2, e2))
    return out

# 後処理
segments_merged = merge_close_segments(segments, max_gap=0.6, min_keep=0.8)
total_dur = librosa.get_duration(path=wav_for_diar)
segments_padded = pad_segments(segments_merged, pad=0.25, total_dur=total_dur)
print(f"segments: raw={len(segments)}, merged={len(segments_merged)}, padded={len(segments_padded)}")

# 話者別にWAV書き出し
y_base, sr_base = librosa.load(wav_for_diar, sr=None, mono=True)
print(f"base audio: sr={sr_base}, dur={len(y_base)/sr_base:.2f}s")

buckets = defaultdict(list)
for spk, s, e in segments_padded:
    buckets[spk].append((s, e))

for spk, spans in buckets.items():
    spkdir = OUTDIR / f"spk_{spk}"
    spkdir.mkdir(exist_ok=True)
    count = 0
    for i, (s, e) in enumerate(spans):
        s0, s1 = int(s*sr_base), int(e*sr_base)
        if s1 > s0:
            seg = y_base[s0:s1]
            sf.write((spkdir / f"{i:02d}_{spk}_{s:.2f}-{e:.2f}.wav").as_posix(), seg, sr_base)
            count += 1
    print(f"{spk}: saved {count} segments in {spkdir}")

In [ ]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from collections import defaultdict

y, sr = sf.read(wav_for_diar, always_2d=False)
if y.ndim > 1:
    y = y.mean(axis=1)

hop = int(0.01 * sr)
win = int(0.032 * sr)
num = max(1, 1 + (len(y) - win) // hop)
pad = max(0, win + (num - 1) * hop - len(y))
if pad > 0:
    y = np.concatenate([y, np.zeros(pad)], axis=0)

rms = np.array([np.sqrt(np.mean(y[i*hop:i*hop+win]**2)) for i in range(num)])
times = (np.arange(num) * hop + win/2) / sr

buckets = defaultdict(list)
for spk, s, e in segments_padded:
    buckets[spk].append((s, e))

plt.figure(figsize=(12, 4))
for spk, spans in sorted(buckets.items()):
    arr = np.zeros_like(rms)
    for s, e in spans:
        mask = (times >= s) & (times < e)
        arr[mask] = rms[mask]
    plt.plot(times, arr, label=spk, linewidth=1.0)

plt.xlabel("Time (s)")
plt.ylabel("Amplitude (RMS)")
plt.title("Speaker-wise Amplitude Over Time")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# === Block S4: 話者クラスタの音響特徴分析 ===
import pandas as pd

# 分析対象のディレクトリ
base_dir = OUTDIR

# 話者ごとの分析結果を格納する辞書
speaker_features = {}

# spk_... という名前のディレクトリをすべて見つける
speaker_dirs = [d for d in base_dir.iterdir() if d.is_dir() and d.name.startswith('spk_')]

print(f"分析対象の話者フォルダ: {[d.name for d in speaker_dirs]}")
print("-" * 30)

# 各話者フォルダをループ
for spk_dir in speaker_dirs:
    speaker_name = spk_dir.name
    
    # この話者の特徴量を一時的に保存するリスト
    all_pitches = []
    all_mfccs = []
    
    # フォルダ内の全WAVファイルを処理
    wav_files = list(spk_dir.glob("*.wav"))
    if not wav_files:
        continue
        
    for wav_file in wav_files:
        y, sr = librosa.load(wav_file, sr=16000)
        
        # --- 1. ピッチ (F0) の抽出 ---
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
        valid_pitches = f0[voiced_flag]
        if len(valid_pitches) > 0:
            all_pitches.extend(valid_pitches)
            
        # --- 2. 音色 (MFCC) の抽出 ---
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        if mfccs.shape[1] > 0:
            all_mfccs.append(np.mean(mfccs, axis=1))

    # この話者の特徴量を集計
    if all_pitches and all_mfccs:
        avg_pitch = np.mean(all_pitches)
        std_pitch = np.std(all_pitches)
        avg_mfcc = np.mean(np.array(all_mfccs), axis=0)
        
        speaker_features[speaker_name] = {
            "avg_pitch": avg_pitch,
            "std_pitch": std_pitch,
            "avg_mfcc": avg_mfcc
        }
        print(f"[{speaker_name}] 平均ピッチ: {avg_pitch:.2f} Hz (標準偏差: {std_pitch:.2f})")

# --- 可視化 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. 平均ピッチの比較 (棒グラフ)
names = list(speaker_features.keys())
avg_pitches = [feat['avg_pitch'] for feat in speaker_features.values()]
std_pitches = [feat['std_pitch'] for feat in speaker_features.values()]
axes[0].bar(names, avg_pitches, yerr=std_pitches, capsize=5, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[0].set_title("比較①：平均ピッチ (声の高さ)")
axes[0].set_ylabel("基本周波数 (F0) [Hz]")
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# 2. 平均MFCCの比較 (線グラフ)
for name, feat in speaker_features.items():
    axes[1].plot(feat['avg_mfcc'], marker='o', linestyle='-', label=name)
axes[1].set_title("比較②：平均MFCC (声の音色・響き)")
axes[1].set_xlabel("MFCC係数")
axes[1].set_ylabel("係数の値")
axes[1].grid(True, linestyle='--', alpha=0.7)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# === Block S5 (改): 話者別に音声ファイルを連結 ===
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path

print("各話者の音声ファイルを連結します...")
print("-" * 30)

# 連結後のファイルを保存するディレクトリ
base_dir = OUTDIR
concat_dir = base_dir / "_concat"
concat_dir.mkdir(exist_ok=True)

# 話者ディレクトリを探す (spk_... という名前のフォルダ)
speaker_dirs = sorted([d for d in base_dir.iterdir() if d.is_dir() and d.name.startswith('spk_')])

# 各話者フォルダをループ
for spk_dir in speaker_dirs:
    speaker_name = spk_dir.name
    
    # フォルダ内にあるWAVファイルを時間順（ファイル名順）に取得
    wav_files = sorted(list(spk_dir.glob("*.wav")))
    
    if not wav_files:
        print(f"話者 {speaker_name} には音声ファイルがありません。スキップします。")
        continue

    print(f"処理中の話者: {speaker_name} ({len(wav_files)}個のファイル)")

    # 連結するための空のリストを準備
    all_audio_segments = []
    
    # サンプリングレートは最初のファイルから取得（すべて同じはず）
    sr = librosa.get_samplerate(wav_files[0])

    # 各音声ファイルを読み込んでリストに追加
    for wav_file in wav_files:
        y, _ = librosa.load(wav_file, sr=sr, mono=True)
        all_audio_segments.append(y)
    
    # NumPyを使ってリスト内の音声をすべて連結
    concatenated_audio = np.concatenate(all_audio_segments)
    
    # 連結した音声ファイルを保存
    output_path = concat_dir / f"{speaker_name}_concatenated.wav"
    sf.write(output_path, concatenated_audio, sr)
    
    print(f"  -> 連結完了: {output_path}")

print("\nすべての処理が完了しました。")